# Retina Vessel Segmentation Training

Run these cells from the `TransUNet` root so relative paths to data, weights, and outputs work as expected.

## 1) Verify environment, data, and pretrained weights
- Checks Python/Torch versions
- Confirms dataset location (`../data/Retina/training/images` and `../data/Retina/training/masks`)
- Confirms pretrained ViT checkpoint path (`../model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz` by default)

In [ ]:
import os
import torch
from pathlib import Path

root = Path('.').resolve()
dataset_root = root / '..' / 'dataset'
dataset_names = [
    "DRIVE",
    "CHASE_DB1",
    "Fundus-AVSeg",
    "HRF",
    "RETA",
    "STARE",
]
train_split = "training"
vit_weights = root / '..' / 'model' / 'vit_checkpoint' / 'imagenet21k' / 'R50+ViT-B_16.npz'

print(f"Working directory: {root}")
print(f"Python version: {os.sys.version.split()[0]}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()} | Current device: {torch.cuda.current_device()}")

if not dataset_root.exists():
    raise SystemExit(f"Dataset root missing: {dataset_root}")

missing = []
for name in dataset_names:
    base = dataset_root / name / train_split
    images_dir = base / 'images'
    masks_dir = base / 'masks'
    ok = images_dir.exists() and masks_dir.exists()
    print(f"{name}: images={images_dir} ({'OK' if images_dir.exists() else 'MISSING'}), "
          f"masks={masks_dir} ({'OK' if masks_dir.exists() else 'MISSING'})")
    if not ok:
        missing.append(name)

print(f"Pretrained ViT: {vit_weights} -> {'OK' if vit_weights.exists() else 'MISSING'}")
if missing:
    raise SystemExit(f"Missing image/mask dirs for: {', '.join(missing)}")
if not vit_weights.exists():
    raise SystemExit("Pretrained ViT weights missing. Place R50+ViT-B_16.npz under ../model/vit_checkpoint/imagenet21k/")


## 2) Launch training (single, unified, or sequential across multiple)
Uses `train_retina.py` with your folder structure under `../dataset/<NAME>/training/{images,masks}`.
- Single dataset: set `--root_path`.
- Unified merged dataset (all roots combined in one DataLoader): set `--unified_roots` with a comma-separated list.
- Sequential (one after another, continuing weights): set `--root_paths` with a comma-separated list.
- To resume from a previous checkpoint, add `--resume path/to/checkpoint.pth`.

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python train_retina.py \
  --unified_roots "../dataset/DRIVE,../dataset/CHASE_DB1,../dataset/Fundus-AVSeg,../dataset/HRF,../dataset/RETA,../dataset/STARE" \
  --train_split training \
  --dataset Retina \
  --vit_name R50-ViT-B_16 \
  --vit_patches_size 16 \
  --img_size 512 \
  --batch_size 24 \
  --max_epochs 150 \
  --max_iterations 30000 \
  --n_skip 3 \
  --num_workers 4

In [ ]:
%%bash
# Example A: single dataset root (add --resume if continuing from a checkpoint)
CUDA_VISIBLE_DEVICES=0 python train_retina.py \
  --root_path ../dataset/DRIVE \
  --train_split training \
  --dataset Retina \
  --vit_name R50-ViT-B_16 \
  --vit_patches_size 16 \
  --img_size 512 \
  --batch_size 24 \
  --max_epochs 150 \
  --max_iterations 30000 \
  --n_skip 3 \
  --num_workers 4 \
  # --resume ../model/.../checkpoint.pth

# Example B: unified training across all six dataset roots (merged into one DataLoader)
# CUDA_VISIBLE_DEVICES=0 python train_retina.py \
#   --unified_roots "../dataset/DRIVE,../dataset/CHASE_DB1,../dataset/Fundus-AVSeg,../dataset/HRF,../dataset/RETA,../dataset/STARE" \
#   --train_split training \
#   --dataset Retina \
#   --vit_name R50-ViT-B_16 \
#   --vit_patches_size 16 \
#   --img_size 512 \
#   --batch_size 24 \
#   --max_epochs 150 \
#   --max_iterations 30000 \
#   --n_skip 3 \
#   --num_workers 4 \
#   # --resume ../model/.../checkpoint.pth

# Example C: sequential across all six dataset roots (one after another)
# CUDA_VISIBLE_DEVICES=0 python train_retina.py \
#   --root_paths "../dataset/DRIVE,../dataset/CHASE_DB1,../dataset/Fundus-AVSeg,../dataset/HRF,../dataset/RETA,../dataset/STARE" \
#   --train_split training \
#   --dataset Retina \
#   --vit_name R50-ViT-B_16 \
#   --vit_patches_size 16 \
#   --img_size 512 \
#   --batch_size 24 \
#   --max_epochs 150 \
#   --max_iterations 30000 \
#   --n_skip 3 \
#   --num_workers 4 \
#   # --resume ../model/.../checkpoint.pth


## 3) (Optional) Run inference on images
Point `--image_dir` or `--image_path` to your test images. Output masks will be saved to `../predictions_retina` by default.

In [ ]:
%%bash
python test_retina.py \
  --image_dir ../dataset/DRIVE/test/images \
  --output_dir ../predictions_retina \
  --dataset Retina \
  --vit_name R50-ViT-B_16 \
  --vit_patches_size 16 \
  --img_size 512 \
  --n_skip 3
